In [1]:
import sys

import os
import glob
import shutil

import numpy as np
import subprocess

import ase.atoms

import arkane.encorr.reference
import arkane.ess
import rmgpy.molecule

sys.path.append(os.environ['DFT_DIR'])
import autotst_wrapper

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun

/home/harris.se/rmg/RMG-Py/rmgpy/rmg/reactors.py:53: RuntimeWarning: Unable to import Julia dependencies, original error: [Errno 2] No such file or directory: 'julia': 'julia'
  warnings.warn("Unable to import Julia dependencies, original error: " + str(e), RuntimeWarning)


Loading DFT database from /work/westgroup/harris.se/autoscience/reaction_calculator/database


In [2]:
# load the reference database
database = arkane.encorr.reference.ReferenceDatabase()
database.load()

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3);
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3);
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual v

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O+1(2)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O+1(2)
ERROR:root:Unable to generate identifier for this molecule:
1 O u0 p3 c-1 {3,S}
2 O u0 p2 c0 {3,D}
3 C u0 p0 c0 {1,S} {2,D} {4,S}
4 H u0 p0 c0 {3,S}

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valen

*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1); O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): S(1); O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): C(3)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted unusual valence(s): O(1)
*** Open Babel Warning  in InChI code
  #1 :Accepted

# Only collect uncharged C,H,O species

In [3]:
# Redo Single Points where geometries don't match corresponding gaussian files
special_set = []
for i in range(len(database.reference_sets['main'])):
    if database.reference_sets['main'][i].charge != 0:
        continue
    if 'N' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'S' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'CL' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'BR' in database.reference_sets['main'][i].smiles.upper():
        continue
    if 'F' in database.reference_sets['main'][i].smiles.upper():
        continue
    special_set.append(i)

# Check progress on geometry optimizations

In [27]:
def has_right_modes(cf):
    if not cf.modes:
        return False
    elif not any(isinstance(mode, rmgpy.statmech.IdealGasTranslation) for mode in cf.modes):
        return False
    elif not any(isinstance(mode, (rmgpy.statmech.LinearRotor, rmgpy.statmech.NonlinearRotor)) for mode in cf.modes):
        return False
    elif not any(isinstance(mode, rmgpy.statmech.HarmonicOscillator) for mode in cf.modes):
        return False
    return True

In [30]:
working_dir = '/scratch/harris.se/guassian_scratch/bac'

incomplete_geo = []
bad_geo = []
complete_geo = []

# for i in range(len(database.reference_sets['main'])):
for i in special_set:
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    if not os.path.exists(sp_logfile):
        incomplete_geo.append(i)
        continue
    try:
        gl = arkane.ess.factory.ess_factory(sp_logfile)
    except arkane.exceptions.LogError:
        incomplete_geo.append(i)
        continue
    
    cf, freqs = gl.load_conformer()
    if not has_right_modes(cf):
        bad_geo.append(i)
    elif gl.load_force_constant_matrix() is None:
        bad_geo.append(i)
    else:
        complete_geo.append(i)

In [31]:
print(len(complete_geo))

120


In [32]:
print(len(bad_geo))

23


In [5]:
autotst_wrapper.check_hessian_cartesian_consistent('/scratch/harris.se/guassian_scratch/bac/species_0038/sp.log')

False

In [6]:
ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][38].adjacency_list)
database_fun.get_unique_species_index(ref_sp)

IndexError: Species C#CC#C not in database

### Add check for consistent hessian ###

In [ ]:
inconsistent = [38]
for i in complete_geo:
    if i in [38]:
        continue
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    print(i)
    try:
        if not autotst_wrapper.check_hessian_cartesian_consistent(sp_logfile):
            inconsistent.append(i)
            print(i, '\tinconsistent')
    except ValueError:
        inconsistent.append(i)
        print(i, '\tFailed')
        

In [ ]:
inconsistent

# Remake geometry optimization files

In [ ]:
# for i in range(len(ref_db.reference_sets['main'])):
my_indices = inconsistent
for i in my_indices:

    # get starting geometry from previous calculations
    preferred_methods = ['ccsd(t)f12', 'cbsqb32023']
    atoms = None
    for preferred_method in preferred_methods:
        for key in database.reference_sets['main'][i].calculated_data.keys():
            comparison = key
            if type(key) != arkane.modelchem.LevelOfTheory:
                comparison = key.energy
            
            if comparison.method == preferred_method:
                syms = database.reference_sets['main'][i].calculated_data[key].xyz_dict['symbols']
                xyz = database.reference_sets['main'][i].calculated_data[key].xyz_dict['coords']
                atoms = ase.Atoms(symbols=syms, positions=xyz)
                break
        if atoms:
            break
    else:
        raise ValueError(f'no preferred level of theory for entry {i}')


    sp_dir = os.path.join(working_dir, f'species_{i:04}')
    os.makedirs(sp_dir, exist_ok=True)
    
    # write the gaussian calculation file
    with open(os.path.join(sp_dir, 'sp.com'), 'w') as f:
        ase.io.gaussian.write_gaussian_in(
            f,
            atoms,
            properties=['energy'],
            method='m062x',
            basis='cc-pvtz',
            mult=database.reference_sets['main'][i].multiplicity,
            charge=database.reference_sets['main'][i].charge,
            opt='calcfc,maxcycles=900',
            freq='',
            extra='IOP(7/33=1,2/9=2000,2/16=3)'
        )

# write the slurm script
run_script = os.path.join(working_dir, 'run.sh')
with open(run_script, 'w') as f:
    f.write("""#!/bin/bash
#SBATCH --job-name=g16_bac_opt
#SBATCH --error=error.log
#SBATCH --nodes=1
#SBATCH --partition=west,short
#SBATCH --exclude=c5003
#SBATCH --mem=20Gb
#SBATCH --time=24:00:00
#SBATCH --cpus-per-task=16
""" +
f'#SBATCH --array={autotst_wrapper.ordered_array_str(my_indices)}%10\n' +
"""
export GAUSS_SCRDIR=/scratch/harris.se/guassian_scratch
mkdir -p $GAUSS_SCRDIR
module load gaussian/g16
source /shared/centos7/gaussian/g16/bsd/g16.profile

RUN_i=$(printf "%04.0f" $(($SLURM_ARRAY_TASK_ID)))

cd "species_${RUN_i}"
g16 sp.com

""")


# Copy completed geometry optimizations from autoscience database

In [ ]:
not_in_db = []
for i in bad_geo:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    try:
        db_index = database_fun.get_unique_species_index(ref_sp)
    except IndexError:
        print(i, ref_sp.smiles, 'not in db')
        not_in_db.append(ref_sp)
        continue

In [9]:
not_in_db = [rmgpy.species.Species(smiles='C#CC#C')]

In [11]:
# database_fun.add_species_to_database(not_in_db)

In [29]:
# for i in bad_geo:
skiplist = [28]
for i in bad_geo:
    ref_sp = rmgpy.species.Species().from_adjacency_list(database.reference_sets['main'][i].adjacency_list)
    try:
        db_index = database_fun.get_unique_species_index(ref_sp)
    except IndexError:
        print(i, ref_sp.smiles, 'not in db')
        not_in_db.append(ref_sp)
        continue
        
    if db_index in skiplist:
        print('skipping', i)
        continue
        
    print(f'Looking for db index {db_index}')
    # see if there's a complete file
    try:
        rotor_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'rotors', 'conformer_*.log'))[0]
    except IndexError:
        rotor_logfile = None
        
    try:
        iop_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'rotors', 'iop_recalc_*.log'))[0]
    except IndexError:
        iop_logfile = None
        
    try:
        arkane_logfile = glob.glob(os.path.join(os.environ['DFT_DIR'], 'thermo', f'species_{db_index:04}', 'arkane', 'conformer_*.log'))[0]
    except IndexError:
        arkane_logfile = None
    
    my_logfile = None
    if iop_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(iop_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = iop_logfile
        except arkane.exceptions.LogError:
            print(f'Bad IOP')
            continue
    elif rotor_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(rotor_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = rotor_logfile
        except arkane.exceptions.LogError:
            print(f'Bad Rotor')
            continue
    elif arkane_logfile is not None:
        try:
            gl = arkane.ess.factory.ess_factory(arkane_logfile)
            cf, freqs = gl.load_conformer()
            if not has_right_modes(cf):
                raise arkane.exceptions.LogError
            elif gl.load_force_constant_matrix() is None:
                arkane.exceptions.LogError
            my_logfile = arkane_logfile
        except arkane.exceptions.LogError:
            print(f'Bad Arkane')
            continue
    else:
        print('nothing to copy')
        continue
    
    dest_file = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    print(f'copying {my_logfile} to {dest_file}')
    shutil.copyfile(my_logfile, dest_file)
    

Looking for db index 1048
nothing to copy
Looking for db index 195
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0195/rotors/conformer_0002.log to /scratch/harris.se/guassian_scratch/bac/species_0005/sp.log
Looking for db index 1049
nothing to copy
Looking for db index 1050
nothing to copy
Looking for db index 1051
nothing to copy
Looking for db index 1052
nothing to copy
Looking for db index 1053
nothing to copy
Looking for db index 1054
nothing to copy
Looking for db index 1055
nothing to copy
Looking for db index 951
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0951/arkane/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0062/sp.log
Looking for db index 954
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0954/arkane/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0072/sp.log
Looking for db index 785
copying /work/westgroup/har

copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0987/arkane/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0188/sp.log
Looking for db index 19
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0019/rotors/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0199/sp.log
Looking for db index 96
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0096/rotors/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0200/sp.log
Looking for db index 5
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0005/rotors/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0201/sp.log
Looking for db index 988
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0988/arkane/conformer_0001.log to /scratch/harris.se/guassian_scratch/bac/species_0202/sp.log
Looking f

copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0186/rotors/conformer_0001.log to /scratch/harris.se/guassian_scratch/bac/species_0351/sp.log
Looking for db index 48
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0048/rotors/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0352/sp.log
Looking for db index 360
nothing to copy
Looking for db index 854
nothing to copy
Looking for db index 1027
nothing to copy
Looking for db index 930
nothing to copy
Looking for db index 365
nothing to copy
Looking for db index 25
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0025/rotors/conformer_0000.log to /scratch/harris.se/guassian_scratch/bac/species_0398/sp.log
Looking for db index 351
nothing to copy
Looking for db index 11
copying /work/westgroup/harris.se/autoscience/reaction_calculator/dft/thermo/species_0011/rotors/conformer_0000.log to /scratch/harris.se/guassi

In [17]:
bad_geo

[4,
 5,
 12,
 15,
 25,
 33,
 45,
 46,
 58,
 62,
 72,
 73,
 74,
 76,
 78,
 83,
 84,
 87,
 88,
 93,
 95,
 96,
 97,
 99,
 100,
 101,
 102,
 103,
 110,
 112,
 113,
 115,
 117,
 125,
 127,
 128,
 150,
 151,
 153,
 154,
 155,
 157,
 159,
 160,
 161,
 162,
 164,
 165,
 184,
 185,
 188,
 199,
 200,
 201,
 202,
 205,
 208,
 209,
 211,
 213,
 214,
 218,
 219,
 220,
 221,
 222,
 224,
 225,
 237,
 239,
 243,
 244,
 248,
 258,
 262,
 264,
 275,
 278,
 279,
 282,
 286,
 288,
 293,
 297,
 301,
 303,
 306,
 309,
 330,
 341,
 342,
 347,
 349,
 350,
 351,
 352,
 353,
 354,
 355,
 360,
 392,
 398,
 399,
 400,
 404,
 405,
 407,
 408,
 409,
 410,
 414,
 417,
 418,
 419,
 420]

In [14]:
bad_geo

[4,
 5,
 12,
 15,
 25,
 33,
 45,
 46,
 58,
 62,
 72,
 73,
 74,
 76,
 78,
 83,
 84,
 87,
 88,
 93,
 95,
 96,
 97,
 99,
 100,
 101,
 102,
 103,
 110,
 112,
 113,
 115,
 117,
 125,
 127,
 128,
 150,
 151,
 153,
 154,
 155,
 157,
 159,
 160,
 161,
 162,
 164,
 165,
 184,
 185,
 188,
 199,
 200,
 201,
 202,
 205,
 208,
 209,
 211,
 213,
 214,
 218,
 219,
 220,
 221,
 222,
 224,
 225,
 237,
 239,
 243,
 244,
 248,
 258,
 262,
 264,
 275,
 278,
 279,
 282,
 286,
 288,
 293,
 297,
 301,
 303,
 306,
 309,
 330,
 341,
 342,
 347,
 349,
 350,
 351,
 352,
 353,
 354,
 355,
 360,
 392,
 398,
 399,
 400,
 404,
 405,
 407,
 408,
 409,
 410,
 414,
 417,
 418,
 419,
 420]

# Check progress on single-point calculations

In [ ]:
incomplete_sp = []
bad_sp = []
complete_sp = []

# for i in range(len(database.reference_sets['main'])):
for i in special_set:
    if i in inconsistent or i in bad_geo:
        continue
    
    
    orca_logfile = os.path.join(working_dir, f'species_{i:04}', 'conformer.out')
    if not os.path.exists(orca_logfile):
        incomplete_sp.append(i)
        continue
    try:
        ol = arkane.ess.factory.ess_factory(orca_logfile)
    except arkane.exceptions.LogError:
        incomplete_sp.append(i)
        continue
        
    # make sure that the coordinates match between orca and Gaussian
    sp_logfile = os.path.join(working_dir, f'species_{i:04}', 'sp.log')
    
    g_coords, g_num, g_mass = arkane.ess.factory.ess_factory(sp_logfile).load_geometry()
    o_coords, o_num, o_mass = ol.load_geometry()
    
    if not np.all(np.equal(np.array(g_coords), np.array(o_coords))):
        g2_coords, g_num, g_mass = arkane.ess.factory.ess_factory(sp_logfile).load_geometry(standard_orientation=True)
        if not np.all(np.equal(np.array(g2_coords), np.array(o_coords))):
            bad_sp.append(i)
            print(f'problem with {i}')
        else:
            complete_sp.append(i)
    else:
        complete_sp.append(i)
    

In [ ]:
incomplete_sp

In [ ]:
complete_sp

In [ ]:
bad_sp

In [ ]:
set(complete_geo) - set(inconsistent)